In [0]:
from pyspark.sql import functions as F

VOLUME = (
    "/Volumes/mindit_training/"
    "bookstore_ec_bronze/bookstore_landing"
) 

SOURCE_PATH = f"{VOLUME}/landing" # Source path for incremental ingestion

TARGET_TABLE = (
    "mindit_training.bookstore_ec_bronze."
    "product_catalogue_raw"
) # Target bronze Delta table for incremental ingestion

CHECKPOINT_PATH = (
    f"{VOLUME}/_checkpoints/product_catalogue"
) # Checkpoint path for incremental ingestion

print("Source:", SOURCE_PATH)
print("Target:", TARGET_TABLE)
print("Checkpoint:", CHECKPOINT_PATH)

Source: /Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/landing
Target: mindit_training.bookstore_ec_bronze.product_catalogue_raw
Checkpoint: /Volumes/mindit_training/bookstore_ec_bronze/bookstore_landing/_checkpoints/product_catalogue


 # Incremental streaming DataFrame for product catalogue using Auto Loader

# Reuse existing schema for Bronze Table excluding technical ingestion columns

In [0]:
source_schema = (
    spark.table(TARGET_TABLE)
    .drop("_source_file", "_ingested_at")
    .schema
) 

df_incremental_catalogue = (
    spark.readStream
    .format("cloudFiles") # Use Auto Loader for incremental ingestion
    .schema(source_schema) 
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "product_catalogue_*.csv") # Only process product catalogue CSV files
    .option("cloudFiles.includeExistingFiles", "false") # Do not include existing files, only new arrivals
    .load(SOURCE_PATH)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("_ingested_at", F.current_timestamp())
) # Incremental streaming DataFrame for product catalogue

print("Auto Loader streaming DataFrame configured.")

Auto Loader streaming DataFrame configured.


# Start the incremental ingestion

In [0]:
query = (
    df_incremental_catalogue.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
) # Start the incremental ingestion query

query.awaitTermination() # Wait for the incremental ingestion query to complete

print(
    "Catalogue records:",
    spark.table(TARGET_TABLE).count()
) # Print the count of catalogue records in the target table

Catalogue records: 8


In [0]:
%sql
-- Run in a Databricks SQL notebook AFTER the silver pipeline has run.
-- Catalog / schema: mindit_training.bookstore_ec_silver

-- 1) Expected after batch 1 only:   2026-10-05 = 295
--    Expected after batch 1 + 2:    2026-10-05 = 365, 2026-10-07 = 85
SELECT business_date, SUM(line_amount) AS pos_net_sales
FROM mindit_training.bookstore_ec_silver.slv_pos_receipt_lines
WHERE receipt_status = 'COMPLETED'
GROUP BY business_date
ORDER BY business_date;

-- 2) Cancelled receipt REC003 is kept with its status but never counted above
SELECT receipt_id, status, receipt_total
FROM mindit_training.bookstore_ec_silver.slv_pos_receipts
ORDER BY receipt_ts;

-- 3) Late file: only REC006 should be flagged after batch 2
SELECT receipt_id, business_date, arrival_date, is_late_arrival
FROM mindit_training.bookstore_ec_silver.slv_pos_receipts
WHERE is_late_arrival;

-- 4) Split payment: REC002 = 50 CARD + 25 CASH = 75
SELECT payment_method, amount
FROM mindit_training.bookstore_ec_silver.slv_pos_payments
WHERE receipt_id = 'REC002';

-- 5) Quarantine should be empty for the fixture
SELECT * FROM mindit_training.bookstore_ec_silver.dq_pos_receipts_quarantine;

-- 6) How many records failed each check (from the pipeline event log)
SELECT
  e.name AS check_name,
  SUM(e.passed_records) AS passed,
  SUM(e.failed_records) AS failed
FROM (
  SELECT explode(from_json(
           details:flow_progress.data_quality.expectations,
           'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS e
  FROM event_log(TABLE(mindit_training.bookstore_ec_silver.slv_pos_receipts_parsed))
  WHERE event_type = 'flow_progress'
)
GROUP BY e.name
ORDER BY e.name;

check_name,passed,failed
file_header_present,5,0
lines_match_total,5,0
receipt_date_matches_file,5,0
receipt_id_present,5,0
receipt_total_valid,5,0
receipt_ts_valid,5,0
status_known,5,0
tenders_match_total,5,0


In [0]:
%sql
SELECT business_date, SUM(line_amount) AS pos_net_sales
FROM mindit_training.bookstore_ec_silver.slv_pos_receipt_lines
WHERE receipt_status = 'COMPLETED'
GROUP BY business_date
ORDER BY business_date;

business_date,pos_net_sales
2026-10-05,295.0000


In [0]:
%sql
SELECT receipt_id, status, receipt_total
FROM mindit_training.bookstore_ec_silver.slv_pos_receipts
ORDER BY receipt_ts;

receipt_id,status,receipt_total
REC001,COMPLETED,90.00
REC002,COMPLETED,75.00
REC003,CANCELLED,200.00
REC004,COMPLETED,100.00
REC005,COMPLETED,30.00


In [0]:
%sql
SELECT receipt_id, business_date, arrival_date, is_late_arrival
FROM mindit_training.bookstore_ec_silver.slv_pos_receipts
WHERE is_late_arrival;

receipt_id,business_date,arrival_date,is_late_arrival


In [0]:
%sql
SELECT payment_method, amount
FROM mindit_training.bookstore_ec_silver.slv_pos_payments
WHERE receipt_id = 'REC002';

payment_method,amount
CARD,50.00
CASH,25.00


In [0]:
%sql
SELECT * FROM mindit_training.bookstore_ec_silver.dq_pos_receipts_quarantine;


receipt_key,store_id,till_id,business_date,receipt_id,status,receipt_total,lines_amount,paid_amount,failed_checks,source_file,_ingested_at,quarantined_at


In [0]:
%sql
SELECT
  e.name AS check_name,
  SUM(e.passed_records) AS passed,
  SUM(e.failed_records) AS failed
FROM (
  SELECT explode(from_json(
           details:flow_progress.data_quality.expectations,
           'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS e
  FROM event_log(TABLE(mindit_training.bookstore_ec_silver.slv_pos_receipts_parsed))
  WHERE event_type = 'flow_progress'
)
GROUP BY e.name
ORDER BY e.name;

check_name,passed,failed
file_header_present,5,0
lines_match_total,5,0
receipt_date_matches_file,5,0
receipt_id_present,5,0
receipt_total_valid,5,0
receipt_ts_valid,5,0
status_known,5,0
tenders_match_total,5,0
